# Concatenación de datos

La idea en esta ocasión es experimentar con JupyterLab, por ende, a diferencia del Proyecto PM2.5 (https://github.com/Albertmestre1256/Calidad-del-aire-en-Montevideo) se intentará crear distintos Notebooks
cada uno con funciones distintas para facilitar su lectura.

`Nota de transparencia`: en este notebook se utilizó IA para comprender el sistema Parquet y además convertir los datos
desde .CSV a Parquet. Al ser algo novedoso para mi, creo importante la aclaración pues no es de mi autoría. Por ende, 
separaré la sección creada con IA de la mía dentro del código.

In [2]:
import pandas as pd
from pathlib import Path

In [5]:
# 1. Definir la ruta de la carpeta donde están los CSV descargados de estudiantes
ruta_carpeta = Path(r"C:\Users\Usuario\Documents\Proyecto de Datos\Proyecto - Ceibal\Datasets & Metadatos\Dataset - Estudiantes")

ruta_salida_carpeta = Path(r"C:\Users\Usuario\Documents\Proyecto de Datos\Proyecto - Ceibal\Recursos\Dataset estudiantes concatenado")
ruta_salida_carpeta.mkdir(parents=True, exist_ok=True)

# 2. Unificación de nombres
equivalencias_columnas = {
    "Cantidad de entregas de tareas en CREA": "Cantidad de entregas de tareas",
    "Cantidad de Comentarios posteados en CREA": "Cantidad de Comentarios posteados",
    "Cantidad de Acciones totales en CREA": "Cantidad de Acciones totales",
    "Cantidad de préstamos en Biblioteca": "Cantidad de préstamos en biblioteca"
}

# 3. Leer los archivos locales y conservar su procedencia
dfs_ceibal = []
errores = []

# Buscar automáticamente todos los archivos CSV en la carpeta de estudiantes
for archivo_csv in ruta_carpeta.glob("*.csv"):
    try:
        df = pd.read_csv(
            archivo_csv,
            sep=None,              # Detecta coma o punto y coma automáticamente
            engine="python",
            encoding="utf-8-sig",
            dtype={
                "Id persona": "string",
                "Grado": "string"
            }
        )

        # Quitar espacios sobrantes en los nombres de columnas
        df.columns = df.columns.str.strip()

        # Unificar nombres antes de concatenar
        df = df.rename(columns=equivalencias_columnas)

        # Identificar el origen
        df["archivo_origen"] = archivo_csv.name
        dfs_ceibal.append(df)

        print(f"Cargado: {archivo_csv.name} | {len(df):,} filas")

    except Exception as e:
        errores.append(archivo_csv.name)
        print(f"Error con {archivo_csv.name}: {e}")

# 4. Unir los archivos cargados
if not dfs_ceibal:
    raise RuntimeError("No se encontró ni pudo cargar ningún archivo CSV en la carpeta de estudiantes.")

ceibal_completo = pd.concat(
    dfs_ceibal,
    ignore_index=True,
    sort=False
)

if errores:
    print("\nATENCIÓN: la carga está incompleta. Archivos fallidos:", errores)

print("\nDimensiones del DataFrame consolidado de estudiantes:", ceibal_completo.shape)

# 5. Exportar el resultado consolidado a la nueva carpeta
ruta_archivo_salida = ruta_salida_carpeta / "dataset_estudiantes_consolidado.parquet"

# A partir de este punto, el código está generado por `IA`
# Guardamos en Parquet para optimizar espacio y velocidad 
# Por otro lado, esta es la primera vez que uso Parquet, por lo que también estoy aprendiendo
ceibal_completo.to_parquet(ruta_archivo_salida, index=False, compression="snappy")
print(f"\n¡Éxito! Dataset de estudiantes consolidado guardado en:\n{ruta_archivo_salida}")

Cargado: actividad-de-estudiantes-2024.csv | 632,002 filas
Cargado: actividad-de-estudiantes-2025.csv | 620,419 filas
Cargado: datos_estudiantes_2019.csv | 668,047 filas
Cargado: datos_estudiantes_2020.csv | 668,113 filas
Cargado: datos_estudiantes_2021.csv | 666,979 filas
Cargado: datos_estudiantes_2022.csv | 657,731 filas
Cargado: estudiantes_datos_abiertos_agesic.csv | 647,420 filas

Dimensiones del DataFrame consolidado de estudiantes: (4560711, 21)

¡Éxito! Dataset de estudiantes consolidado guardado en:
C:\Users\Usuario\Documents\Proyecto de Datos\Proyecto - Ceibal\Recursos\Dataset estudiantes concatenado\dataset_estudiantes_consolidado.parquet


In [6]:
# 1. Definir la ruta de la carpeta donde están los CSV de docentes
ruta_carpeta = Path(r"C:\Users\Usuario\Documents\Proyecto de Datos\Proyecto - Ceibal\Datasets & Metadatos\Dataset - Docentes")

ruta_salida_carpeta = Path(r"C:\Users\Usuario\Documents\Proyecto de Datos\Proyecto - Ceibal\Recursos\Dataset docentes concatenado")
ruta_salida_carpeta.mkdir(parents=True, exist_ok=True)

# 2. Unificar nombres de columnas (complétalo si notas diferencias entre los años)
equivalencias_columnas = {}

# 3. Leer los archivos locales y conservar su procedencia
dfs_docentes = []
errores = []

# Buscar automáticamente todos los CSV en la carpeta de docentes
for archivo_csv in ruta_carpeta.glob("*.csv"):
    try:
        df = pd.read_csv(
            archivo_csv,
            sep=None,              # Detecta separador automáticamente
            engine="python",
            encoding="utf-8-sig",
            dtype={
                "Id persona": "string" # Forzamos el ID a texto para no perder ceros a la izquierda ni tener errores de formato
            }
        )

        # Quitar espacios invisibles al principio/final de los nombres de columnas
        df.columns = df.columns.str.strip()

        # Renombrar columnas si agregaste algo al diccionario de arriba
        if equivalencias_columnas:
            df = df.rename(columns=equivalencias_columnas)

        # Identificar de qué año/archivo viene cada fila
        df["archivo_origen"] = archivo_csv.name
        dfs_docentes.append(df)

        print(f"Cargado: {archivo_csv.name} | {len(df):,} filas")

    except Exception as e:
        errores.append(archivo_csv.name)
        print(f"Error con {archivo_csv.name}: {e}")

# 4. Unir los archivos cargados
if not dfs_docentes:
    raise RuntimeError("No se encontró ni pudo cargar ningún archivo CSV en la carpeta de Docentes.")

docentes_completo = pd.concat(
    dfs_docentes,
    ignore_index=True,
    sort=False
)

if errores:
    print("\nATENCIÓN: la carga está incompleta. Archivos fallidos:", errores)

print("\nDimensiones del DataFrame consolidado de docentes:", docentes_completo.shape)

# 5. Exportar a formato Parquet en la nueva carpeta
ruta_archivo_salida = ruta_salida_carpeta / "dataset_docentes_consolidado.parquet"

docentes_completo.to_parquet(ruta_archivo_salida, index=False, compression="snappy")
print(f"\n¡Éxito! Dataset de docentes consolidado guardado en:\n{ruta_archivo_salida}")

Cargado: actividad-de-docentes-2024.csv | 55,165 filas
Cargado: actividad-de-docentes-2025.csv | 56,074 filas
Cargado: datos_docentes_2019.csv | 51,370 filas
Cargado: datos_docentes_2020.csv | 50,122 filas
Cargado: datos_docentes_2021.csv | 52,393 filas
Cargado: datos_docentes_2022.csv | 54,495 filas
Cargado: docentes_datos_abiertos_agesic.csv | 59,534 filas

Dimensiones del DataFrame consolidado de docentes: (379153, 14)

¡Éxito! Dataset de docentes consolidado guardado en:
C:\Users\Usuario\Documents\Proyecto de Datos\Proyecto - Ceibal\Recursos\Dataset docentes concatenado\dataset_docentes_consolidado.parquet
